In [425]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler,LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.impute import SimpleImputer

In [426]:
df = pd.read_csv('datasets/RTA Dataset.csv')

In [427]:
df.sample(5)

,Time,Day_of_week,Age_band_of_driver,Sex_of_driver,Educational_level,Vehicle_driver_relation,Driving_experience,Type_of_vehicle,Owner_of_vehicle,Service_year_of_vehicle,...,Vehicle_movement,Casualty_class,Sex_of_casualty,Age_band_of_casualty,Casualty_severity,Work_of_casuality,Fitness_of_casuality,Pedestrian_movement,Cause_of_accident,Accident_severity
8056,20:00:00,Tuesday,31-50,Male,Junior high school,Employee,Below 1yr,Stationwagen,Owner,NaN,...,Going straight,Pedestrian,Female,18-30,2,NaN,NaN,Unknown or other,No distancing,Slight Injury
3217,15:15:00,Monday,18-30,Male,Junior high school,Employee,5-10yr,Other,Owner,Above 10yr,...,Turnover,na,na,na,na,Self-employed,Normal,Not a Pedestrian,Moving Backward,Slight Injury
11138,6:12:00,Saturday,Over 51,Male,Junior high school,Employee,1-2yr,Taxi,Owner,Unknown,...,Going straight,na,na,na,na,Driver,Normal,Not a Pedestrian,Driving carelessly,Slight Injury
7049,8:30:00,Wednesday,18-30,Male,Elementary school,Employee,Below 1yr,Public (13?45 seats),Owner,Unknown,...,Getting off,na,na,na,na,Driver,Normal,Not a Pedestrian,Changing lane to the right,Slight Injury
3130,22:30:00,Tuesday,18-30,Male,Junior high school,Employee,2-5yr,Lorry (41?100Q),Owner,Unknown,...,Getting off,na,na,na,na,Driver,Normal,Not a Pedestrian,No distancing,Slight Injury


In [428]:
df.isnull().mean()*100

Time                            0.000000
Day_of_week                     0.000000
Age_band_of_driver              0.000000
Sex_of_driver                   0.000000
Educational_level               6.016564
Vehicle_driver_relation         4.701202
Driving_experience              6.731082
Type_of_vehicle                 7.713543
Owner_of_vehicle                3.913608
Service_year_of_vehicle        31.893472
Defect_of_vehicle              35.945112
Area_accident_occured           1.940565
Lanes_or_Medians                3.126015
Road_allignment                 1.152972
Types_of_Junction               7.202014
Road_surface_type               1.396557
Road_surface_conditions         0.000000
Light_conditions                0.000000
Weather_conditions              0.000000
Type_of_collision               1.258525
Number_of_vehicles_involved     0.000000
Number_of_casualties            0.000000
Vehicle_movement                2.500812
Casualty_class                  0.000000
Sex_of_casualty 

In [429]:
# Convert 'Time' to hour (numeric)
df['Time'] = pd.to_datetime(df['Time'],  format="%H:%M:%S", errors='coerce').dt.hour

In [430]:
# Features and target
X = df.drop(columns=['Accident_severity'])
Y = df['Accident_severity']

In [431]:
#check in case of missing values in Y (output)
#mask = y.notna()
#x= x[mask]
#y= y[mask]

In [432]:
# Split data
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)


In [433]:
# Define column types
numerical_cols = ['Number_of_vehicles_involved', 'Number_of_casualties','Time']
ordinal_cols = [
    'Age_band_of_driver', 'Driving_experience', 'Educational_level',
    
    'Service_year_of_vehicle', 'Age_band_of_casualty', 'Casualty_severity'
]
#y(output) for label encoding 
label_cols=['Accident_severity'] 
nominal_cols = list(set(X.columns) - set(numerical_cols) - set(ordinal_cols)-set(label_cols))

In [434]:
#make spereate pipeline for each type columns 
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

categorical_nominal_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('OneHotencoding',OneHotEncoder(handle_unknown='ignore'))
])
categorical_ordinal_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('OrdinalEncoding',OrdinalEncoder())
])

In [435]:
#for y(output)
le = LabelEncoder()
le.fit(y_train)
Y_train_encoded = le.transform(Y_train)
Y_test_encoded = le.transform(Y_test)

In [436]:
Y_test_encoded

array([2, 2, 1, ..., 2, 2, 2], shape=(2464,))

In [437]:
y_test.shape

(2464,)

In [438]:
#creating Columns Transformer
preprocessor = ColumnTransformer(
    transformers=[
        ('numerical_side', numerical_transformer, numerical_cols),
        ('categorical_nominal_side', categorical_nominal_transformer, nominal_cols),
        ('categorical_ordinal_side', categorical_ordinal_transformer,ordinal_cols)
    ]
)

In [439]:
#creating pipeline
clf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000))
])

In [440]:
from sklearn import set_config

set_config(display='diagram')
clf

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numerical_side',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer()),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Number_of_vehicles_involved',
                                                   'Number_of_casualties',
                                                   'Time']),
                                                 ('categorical_nominal_side',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('OneHotencoding',
                                                                   OneHotEncoder(han...
                                                   'Work_of_casuality',
                                                   'Type_of_vehicle']),
                                                 ('categorical_ordinal_side',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('OrdinalEncoding',
                                                                   OrdinalEncoder())]),
                                                  ['Age_band_of_driver',
                                                   'Driving_experience',
                                                   'Educational_level',
                                                   'Service_year_of_vehicle',
                                                   'Age_band_of_casualty',
                                                   'Casualty_severity'])])),
                ('classifier', LogisticRegression(max_iter=1000))])

In [441]:
clf.fit(X_train,Y_train_encoded)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numerical_side',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer()),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Number_of_vehicles_involved',
                                                   'Number_of_casualties',
                                                   'Time']),
                                                 ('categorical_nominal_side',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('OneHotencoding',
                                                                   OneHotEncoder(han...
                                                   'Work_of_casuality',
                                                   'Type_of_vehicle']),
                                                 ('categorical_ordinal_side',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('OrdinalEncoding',
                                                                   OrdinalEncoder())]),
                                                  ['Age_band_of_driver',
                                                   'Driving_experience',
                                                   'Educational_level',
                                                   'Service_year_of_vehicle',
                                                   'Age_band_of_casualty',
                                                   'Casualty_severity'])])),
                ('classifier', LogisticRegression(max_iter=1000))])

In [442]:
#predection
Y_pred = clf.predict(X_test)

In [443]:
print('Accuracy Score:',accuracy_score(Y_test_encoded,Y_pred))

Accuracy Score: 0.8372564935064936


# Cross Validation using Pipeline

In [444]:
# cross validation using cross_val_score
from sklearn.model_selection import cross_val_score
cross_val_score(clf, X_train, Y_train_encoded, cv=5, scoring='accuracy').mean()

np.float64(0.8474422270124933)

In [445]:
#to choose best parameters
param_grid = {
    'preprocessor__numerical_side__imputer__strategy': ['mean', 'median'],
    'preprocessor__categorical_nominal_side__imputer__strategy': ['most_frequent', 'constant'],
    'preprocessor__categorical_ordinal_side__imputer__strategy': ['most_frequent', 'constant'],
    'classifier__C': [0.1, 1.0, 10, 100]
}

In [446]:
grid_search = GridSearchCV(clf, param_grid, cv=5)
grid_search.fit(X_train, Y_train_encoded)
print("Best parameters:")
print(grid_search.best_params_)

Best parameters:
{'classifier__C': 0.1, 'preprocessor__categorical_nominal_side__imputer__strategy': 'most_frequent', 'preprocessor__categorical_ordinal_side__imputer__strategy': 'most_frequent', 'preprocessor__numerical_side__imputer__strategy': 'mean'}
